# GP4 Qwen2.5-7B Unsloth QLoRA Pilot

Use only after handwritten seed review passes. Do not add secrets to notebook source. Store `HF_TOKEN` in Colab Secrets.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/gp4_finetune_factory'
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
!python -m pip install -q unsloth datasets trl

In [ ]:
from pathlib import Path
split_path = Path(PROJECT_DIR) / 'data/splits/train.jsonl'
if not split_path.exists():
    raise FileNotFoundError('Run validation, dedupe, and split building before training.')
print(split_path)

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-7B-Instruct',
    max_seq_length=2048,
    load_in_4bit=True,
    token=HF_TOKEN,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
dataset = load_dataset('json', data_files=str(split_path), split='train')

def format_row(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False)}

dataset = dataset.map(format_row, remove_columns=dataset.column_names)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    args=SFTConfig(
        output_dir=f'{PROJECT_DIR}/models/qwen25_gp4_lora_pilot',
        max_steps=100,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=5,
        save_steps=25,
    ),
)
trainer.train()
trainer.save_model(f'{PROJECT_DIR}/models/qwen25_gp4_lora_pilot')